In [1]:
import pandas as pd
import numpy as np


# CSV dosyasını oku
df = pd.read_csv("../../data/activity_recommendations.csv")

In [2]:
print(df["activity_type"].value_counts())
print(df["general_suitability_score"].value_counts())
#kolon kontrolü
print(df.columns.tolist())
print("\nShape:", df.shape)
print("\nNull values:\n", df.isnull().sum())
print("\nActivity counts:\n", df["activity_type"].value_counts())

activity_type
commute         403
outdoor_work    389
walking         380
cycling         373
sports          373
picnic          367
driving         360
running         355
Name: count, dtype: int64
general_suitability_score
10    879
9     538
6     460
5     295
4     237
8     232
0     209
3      78
7      39
2      28
1       5
Name: count, dtype: int64
['rec_id', 'station_id', 'station_name', 'timestamp', 'month', 'hour_of_day', 'day_of_week', 'is_weekend', 'season', 'weather_condition', 'temperature_c', 'feels_like_c', 'humidity_pct', 'wind_speed_kmh', 'precipitation_mm', 'cloud_cover_pct', 'uv_index', 'activity_type', 'general_suitability_score', 'activity_suitability_score', 'umbrella_needed', 'clothing_recommendation', 'activity_advice', 'go_or_no']

Shape: (3000, 24)

Null values:
 rec_id                        0
station_id                    0
station_name                  0
timestamp                     0
month                         0
hour_of_day                   0
day

In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor


df = pd.read_csv("../../data/activity_recommendations.csv")
df.head()

#activity_type için
#feature belirleme
features = [
    "month",
    "is_weekend",
    "season",
    "weather_condition",
    "temperature_c",
    "feels_like_c",
    "humidity_pct",
    "wind_speed_kmh",
    "precipitation_mm",
    "cloud_cover_pct",
    "uv_index",
    "hour_of_day"
]

categorical_cols = [
    "season",
    "weather_condition"
]

numeric_cols = [col for col in features if col not in categorical_cols]

X = df[features].copy()

In [4]:
#Preprocessor(ne olduğuna bak)
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols
        )
    ]
)

In [5]:
#go_or_no model
y_go = df["go_or_no"]

X_train_go, X_test_go, y_train_go, y_test_go = train_test_split(
    X, y_go, test_size=0.2, random_state=42, stratify=y_go
)

go_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

go_model.fit(X_train_go, y_train_go)

go_preds = go_model.predict(X_test_go)

print("GO_OR_NO MODEL")
print("Accuracy:", accuracy_score(y_test_go, go_preds))
print("F1:", f1_score(y_test_go, go_preds))
df["go_or_no"].value_counts(normalize=True)


# 1. OneHotEncoder'ı al
ohe = go_model.named_steps["preprocessor"] \
              .named_transformers_["cat"] \
              .named_steps["onehot"]

# 2. Encoded categorical feature isimleri
encoded_cat_names = ohe.get_feature_names_out(categorical_cols)

# 3. Tüm feature isimlerini birleştir
all_feature_names = np.concatenate([encoded_cat_names, numeric_cols])

# 4. Feature importance al
importances = go_model.named_steps["model"].feature_importances_

# 5. DataFrame oluştur
feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

# 6. Göster
print("\n=== GO_OR_NO Feature Importance ===")
feature_importance.head(20)


GO_OR_NO MODEL
Accuracy: 0.9683333333333334
F1: 0.9803516028955532

=== GO_OR_NO Feature Importance ===


,feature,importance
6,weather_condition_hail,0.286076
5,weather_condition_drizzle,0.106032
17,humidity_pct,0.088562
3,weather_condition_clear,0.082941
2,season_winter,0.081076
19,precipitation_mm,0.056047
13,month,0.055857
7,weather_condition_heavy_rain,0.030025
18,wind_speed_kmh,0.029694
14,is_weekend,0.023825


In [6]:
#confusion matrix for go_or_no
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test_go, go_preds))


[[107  17]
 [  2 474]]


In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# =========================================================
# IMPORTANT:
# X içinde go_or_no ve activity_suitability_score OLMAMALI
# X sadece weather + time + location feature'larını içermeli
# =========================================================

activity_models = {}
activity_metrics = []
activity_thresholds = {}

activities = sorted(df["activity_type"].dropna().unique())

for activity in activities:
    print(f"\nTraining model for: {activity}")

    # One-vs-rest target
    y_activity = (df["activity_type"] == activity).astype(int)

    positive_count = y_activity.sum()
    negative_count = len(y_activity) - positive_count

    if positive_count < 10:
        print(f"Skipping {activity} because positive samples are too few: {positive_count}")
        continue

    # imbalance için dinamik weight
    scale_pos_weight = negative_count / positive_count

    X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
        X, y_activity, test_size=0.2, random_state=42, stratify=y_activity
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42
        ))
    ])

    model.fit(X_train_a, y_train_a)

    # probability al
    probs = model.predict_proba(X_test_a)[:, 1]

    # en iyi threshold'u F1'a göre seç
    best_f1 = 0
    best_threshold = 0.5
    best_preds = None

    for t in np.arange(0.10, 0.51, 0.05):
        preds_t = (probs >= t).astype(int)
        f1_t = f1_score(y_test_a, preds_t, zero_division=0)

        if f1_t > best_f1:
            best_f1 = f1_t
            best_threshold = t
            best_preds = preds_t


 
    
    # final metrics
    acc = accuracy_score(y_test_a, best_preds)
    prec = precision_score(y_test_a, best_preds, zero_division=0)
    rec = recall_score(y_test_a, best_preds, zero_division=0)
    f1 = f1_score(y_test_a, best_preds, zero_division=0)

    activity_models[activity] = model
    activity_thresholds[activity] = best_threshold

   

    activity_metrics.append({
        "activity": activity,
        "positive_count": int(positive_count),
        "scale_pos_weight": round(scale_pos_weight, 2),
        "best_threshold": round(float(best_threshold), 2),
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1": round(float(f1), 4)
    })

    activity_thresholds["walking"] = 0.17
    activity_thresholds["commute"] = 0.30


metrics_df = pd.DataFrame(activity_metrics).sort_values(
    by=["f1", "recall"], ascending=[False, False]
).reset_index(drop=True)

metrics_df


Training model for: commute

Training model for: cycling

Training model for: driving

Training model for: outdoor_work

Training model for: picnic

Training model for: running

Training model for: sports

Training model for: walking


,activity,positive_count,scale_pos_weight,best_threshold,accuracy,precision,recall,f1
0,walking,380,6.89,0.40,0.6067,0.1667,0.5263,0.2532
1,outdoor_work,389,6.71,0.30,0.4433,0.1464,0.6795,0.2409
2,commute,403,6.44,0.10,0.1700,0.1361,0.9630,0.2385
3,sports,373,7.04,0.30,0.4717,0.1441,0.6533,0.2361
4,cycling,373,7.04,0.10,0.1717,0.1298,0.9867,0.2295
5,running,355,7.45,0.25,0.3383,0.1295,0.8028,0.2231
6,driving,360,7.33,0.35,0.5100,0.1349,0.5694,0.2181
7,picnic,367,7.17,0.15,0.2033,0.1229,0.9041,0.2164


In [8]:
def rank_activities_for_input(input_df, activity_models, activity_thresholds, go_model):
    import pandas as pd

    input_df = input_df[features].copy()

    go_pred = go_model.predict(input_df)[0]
    go_prob = go_model.predict_proba(input_df)[0][1]

    gate = 0.3 + 0.7 * go_prob

    rain = input_df["precipitation_mm"].iloc[0]
    wind = input_df["wind_speed_kmh"].iloc[0]
    hour = input_df["hour_of_day"].iloc[0]
    temp = input_df["temperature_c"].iloc[0]

    results = []

    for activity, model in activity_models.items():
        activity_prob = model.predict_proba(input_df)[0][1]
        threshold = activity_thresholds.get(activity, 0.5)

        relative_activity_score = activity_prob / threshold if threshold > 0 else activity_prob
        relative_activity_score = min(relative_activity_score, 1.3)

        base_score = (
            0.85 * activity_prob +
            0.15 * (relative_activity_score / 1.5)
        ) * gate

        adjustment = 1.0

        if activity == "walking":
            if rain < 2 and wind < 20 and hour < 22:
                adjustment = 1.45
            else:
                adjustment = 0.90

        elif activity == "driving":
            if rain >= 5 or hour >= 21:
                adjustment = 1.15
            else:
                adjustment = 0.75

        elif activity == "cycling":
            if rain >= 3 or wind >= 20 or hour >= 21:
                adjustment = 0.65

        elif activity == "picnic":
            if rain >= 2 or hour >= 20:
                adjustment = 0.60
            elif rain == 0 and 15 <= temp <= 28:
                adjustment = 1.3

        elif activity == "commute":
            adjustment = 1.00

        final_score = base_score * adjustment

        results.append({
            "activity": activity,
            "activity_probability": round(float(activity_prob), 4),
            "threshold": round(float(threshold), 2),
            "relative_activity_score": round(float(relative_activity_score), 4),
            "go_or_no_pred": int(go_pred),
            "go_or_no_probability": round(float(go_prob), 4),
            "gate": round(float(gate), 4),
            "adjustment": round(float(adjustment), 2),
            "final_score": round(float(final_score), 4)
        })

    return pd.DataFrame(results).sort_values(by="final_score", ascending=False).reset_index(drop=True)

In [9]:
input_a = pd.DataFrame([{
    "station_name": "Izmir",
    "month": 4,
    "hour_of_day": 15,
    "day_of_week": 3,
    "is_weekend": 0,
    "season": "spring",
    "weather_condition": "clear",
    "temperature_c": 18.0,
    "feels_like_c": 18.0,
    "humidity_pct": 55.0,
    "wind_speed_kmh": 10.0,
    "precipitation_mm": 0.0,
    "cloud_cover_pct": 20.0,
    "uv_index": 5.0,
    "hour_of_day": 23
}])

input_b = pd.DataFrame([{
    "station_name": "Izmir",
    "month": 1,
    "hour_of_day": 22,
    "day_of_week": 6,
    "is_weekend": 1,
    "season": "winter",
    "weather_condition": "heavy_rain",
    "temperature_c": 4.0,
    "feels_like_c": 1.0,
    "humidity_pct": 95.0,
    "wind_speed_kmh": 30.0,
    "precipitation_mm": 8.0,
    "cloud_cover_pct": 100.0,
    "uv_index": 0.0,
    "hour_of_day": 23
}])

ranked_a = rank_activities_for_input(
    input_df=input_a,
    activity_models=activity_models,
    activity_thresholds=activity_thresholds,
    go_model=go_model
)

ranked_b = rank_activities_for_input(
    input_df=input_b,
    activity_models=activity_models,
    activity_thresholds=activity_thresholds,
    go_model=go_model
)

display(ranked_a)

display(ranked_b)

,activity,activity_probability,threshold,relative_activity_score,go_or_no_pred,go_or_no_probability,gate,adjustment,final_score
0,driving,0.4607,0.35,1.3000,1,1.0,1.0,1.15,0.5999
1,running,0.4736,0.25,1.3000,1,1.0,1.0,1.00,0.5326
2,outdoor_work,0.4319,0.30,1.3000,1,1.0,1.0,1.00,0.4971
3,sports,0.3259,0.30,1.0863,1,1.0,1.0,1.00,0.3856
4,cycling,0.4095,0.10,1.3000,1,1.0,1.0,0.65,0.3107
5,commute,0.2467,0.30,0.8224,1,1.0,1.0,1.00,0.2919
6,walking,0.2095,0.17,1.2326,1,1.0,1.0,0.90,0.2712
7,picnic,0.3597,0.15,1.3000,1,1.0,1.0,0.60,0.2614


,activity,activity_probability,threshold,relative_activity_score,go_or_no_pred,go_or_no_probability,gate,adjustment,final_score
0,driving,0.4609,0.35,1.3000,0,0.1967,0.4377,1.15,0.2626
1,walking,0.3827,0.17,1.3000,0,0.1967,0.4377,0.90,0.1793
2,commute,0.1630,0.30,0.5434,0,0.1967,0.4377,1.00,0.0844
3,outdoor_work,0.1430,0.30,0.4768,0,0.1967,0.4377,1.00,0.0741
4,cycling,0.1466,0.10,1.3000,0,0.1967,0.4377,0.65,0.0724
5,picnic,0.1748,0.15,1.1656,0,0.1967,0.4377,0.60,0.0696
6,running,0.0779,0.25,0.3115,0,0.1967,0.4377,1.00,0.0426
7,sports,0.0376,0.30,0.1254,0,0.1967,0.4377,1.00,0.0195


In [10]:
#Top activity listesini alma
top_n = 5
top_activities1 = ranked_a.head(top_n)["activity"].tolist()
print(top_activities1)

top_n = 5
top_activities2 = ranked_b.head(top_n)["activity"].tolist()
print(top_activities2)

['driving', 'running', 'outdoor_work', 'sports', 'cycling']
['driving', 'walking', 'commute', 'outdoor_work', 'cycling']


In [11]:
joblib.dump(go_model, "../../models/go_model.pkl")
joblib.dump(activity_models, "../../models/activity_models.pkl")
joblib.dump(activity_thresholds, "../../models/activity_thresholds.pkl")

['../../models/activity_thresholds.pkl']